# **1. Without Langchain - Do we had HARNESS?**

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")

# **1.1 Building a Plain Model**

In [ ]:
from openai import OpenAI
client = OpenAI()

SYSTEM_PROMPT = "You are a helpful assistant."

response = client.responses.create(model="gpt-4o-mini",
                                   instructions=SYSTEM_PROMPT,
                                   input="What's the weather in San Francisco?")
print(response.output_text)

# **1.2 Now give the model a tool**

In [ ]:
from openai import OpenAI
import json

client = OpenAI()

SYSTEM_PROMPT = "You are a helpful assistant."

def get_weather(city: str) -> str:
    return f"It's always sunny in {city}!"

tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get weather for a given city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string"
                }
            },
            "required": ["city"],
            "additionalProperties": False
        },
        "strict": True
    }
]

response = client.responses.create(
    model="gpt-4o-mini",
    instructions=SYSTEM_PROMPT,
    input="What's the weather in San Francisco?",
    tools=tools
)

print(response.output)

# **IMPORTANT NOTICE**

1. The model **"did NOT" execute your Python function**.

**Then..?**
* It only **requested** that your application should execute it.

# This is the exact point where researchers introduced the word **"Harness"**.

# =============================================================================

# **1.3 Now let's build with Harness**

In [ ]:
from openai import OpenAI
import json

client = OpenAI()

SYSTEM_PROMPT = "You are a helpful assistant."

#Step 1: Create the Tool
def get_weather(city: str) -> str:
    return f"It's always sunny in {city}!"

#Step 2: Create the Tool Definition
tools = [{"type": "function",
          "name": "get_weather",
          "description": "Get weather for a given city.",
          "parameters": {"type": "object",
                         "properties": {"city": {"type": "string"}},
                         "required": ["city"],
                         "additionalProperties": False},
        "strict": True}]

#Step 3 - Create the Tool Map Registry
tool_map = {"get_weather": get_weather}

#Step 4- Get the User Request :
input_items = [{"role": "user","content": "What's the weather in San Francisco?"}]


#Step 5: Create AGENT HARNESS
while True:
    # 5.1. Ask the model what to do
    response = client.responses.create(model="gpt-4o-mini",
                                       instructions=SYSTEM_PROMPT,
                                       input=input_items,
                                       tools=tools)

    # 5.2. Preserve the model's response
    input_items += response.output

    # 5.3. Check whether the model requested a tool
    tool_calls = [item for item in response.output if item.type == "function_call"]

    # 5.4. No tool call = final answer
    if not tool_calls:
        print(response.output_text)
        break

    # 5.5. Execute requested tools
    for tool_call in tool_calls:

        print("MODEL → TOOL")
        print("Tool:", tool_call.name)
        print("Arguments:", tool_call.arguments)

        arguments = json.loads(tool_call.arguments)

        tool_function = tool_map[tool_call.name]

        result = tool_function(**arguments)

        print("TOOL → MODEL")
        print("Result:", result)

        # 5.6. We give the tool's result back to the model.
        input_items.append({"type": "function_call_output",
                            "call_id": tool_call.call_id,
                            "output": result})

# **Interview Question 1**

## **What is Agent Loop?.**

**"while True:"**

means:

**Do it again until the model no longer asks for a tool.**

That is your agent loop.

# Till here is **WITHOUT Langchain version** of Building Harness.

And for the question, "Can we build Harness without any Agentic AI Frameworks like Langchain?"

The Answer is...

# =============================================================================

# **2. LangChain Version - Building Harness**

In [ ]:
pip install -qU langchain "langchain[openai]"

In [ ]:
from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="openai:gpt-5.5",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
)
print(result["messages"][-1].content)

# **The Important Lesson...?**

Previously, we wrote the harness ourselves. Now LangChain is giving us that harness.